# CreditWise — Model Explainability

**Notebook 04 of 04**

This notebook presents SHAP (SHapley Additive exPlanations) analysis for the best model (XGBoost):
- Global feature importance
- SHAP summary (beeswarm) plot
- Local explanation for an individual applicant
- Probability calibration results
- Fairness analysis

> **Important**: SHAP values explain how features contributed to the *model's prediction*.
> They do NOT establish that a feature *caused* the real-world outcome.
> SHAP values for tree models are in the log-odds (margin) space.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from pathlib import Path
from IPython.display import display, Image
from sklearn.model_selection import train_test_split

from src.config import (
    BEST_MODEL_FILE, PREPROCESSING_PIPELINE_FILE, FEATURE_LIST_FILE,
    MODEL_METADATA_FILE, FIGURES_DIR, RANDOM_SEED, TARGET_COLUMN,
)
from src.data_loader import load_raw_data, clean_raw_data
from src.feature_engineering import engineer_features, get_all_feature_names
from src.explainability import build_explainer, compute_shap_values, explain_single

plt.rcParams.update({'figure.facecolor':'#0f172a','axes.facecolor':'#1e293b',
                     'axes.labelcolor':'#e2e8f0','xtick.color':'#94a3b8',
                     'ytick.color':'#94a3b8','text.color':'#e2e8f0','figure.dpi':110})
print('Ready.')

In [ ]:
# Load artefacts
pipeline = joblib.load(PREPROCESSING_PIPELINE_FILE)
model    = joblib.load(BEST_MODEL_FILE)
with open(FEATURE_LIST_FILE) as f:
    feature_names = json.load(f)
with open(MODEL_METADATA_FILE) as f:
    meta = json.load(f)
model_name = meta['best_model_name']
print(f'Model: {model_name}  |  Features: {len(feature_names)}')

# Reload test split
df = clean_raw_data(load_raw_data())
df_eng = engineer_features(df)
X = df_eng[feature_names]
y = df_eng[TARGET_COLUMN]
_, X_test, _, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y)
_, X_train, _, _ = train_test_split(X, y, test_size=0.80, random_state=RANDOM_SEED, stratify=y)
X_test_tf  = pipeline.transform(X_test)
X_train_tf = pipeline.transform(X_train)

## 1. Global SHAP Feature Importance

In [ ]:
imp_path = Path('../reports/results/shap_importance.csv')
if imp_path.exists():
    imp_df = pd.read_csv(imp_path)
    print(f'Top 15 features by mean |SHAP| ({model_name}):')
    display(imp_df.head(15).style.background_gradient(subset=['mean_abs_shap'], cmap='Blues')
            .format({'mean_abs_shap': '{:.4f}'}))
else:
    print('shap_importance.csv not found — run src/post_train_analysis.py first.')

In [ ]:
bar_path = FIGURES_DIR / f'shap_bar_importance_xgboost.png'
if bar_path.exists():
    display(Image(str(bar_path)))

## 2. SHAP Beeswarm Summary Plot

In [ ]:
sum_path = FIGURES_DIR / 'shap_summary_xgboost.png'
if sum_path.exists():
    display(Image(str(sum_path)))
    print('Interpretation:')
    print('  • Each dot = one test sample')
    print('  • Colour = feature value (red=high, blue=low)')
    print('  • X-axis position = SHAP value (right → increases predicted risk)')
    print('  • SHAP values are in log-odds (margin) space — sign/rank interpretable')

## 3. Key SHAP Findings

In [ ]:
findings = [
    ('total_past_due', 'HIGHEST importance. Cumulative delinquency across all severity levels is the strongest default predictor.'),
    ('RevolvingUtilizationOfUnsecuredLines', 'Very high revolving credit usage strongly increases predicted risk.'),
    ('age', 'Younger borrowers show higher predicted risk; older borrowers lower.'),
    ('DebtRatio', 'Higher debt ratio increases predicted risk.'),
    ('credit_line_density', 'More credit lines relative to credit age increases predicted risk.'),
    ('MonthlyIncome', 'Higher income reduces predicted risk (income_per_dependent also shows this).'),
]
print('Key SHAP findings for XGBoost (from global analysis):')
for feat, finding in findings:
    print(f'\n  [{feat}]')
    print(f'  {finding}')
print('\nNote: These are model-level observations, not causal claims.')

## 4. Local Explanation — Individual Applicant

In [ ]:
# High-risk applicant example (choose a true positive from the test set)
y_test_arr = y_test.values
y_prob = model.predict_proba(X_test_tf)[:, 1]

# Find a high-probability true positive (actual default, model predicts high risk)
tp_mask = (y_test_arr == 1) & (y_prob > 0.6)
tp_idx = np.where(tp_mask)[0]
if len(tp_idx) > 0:
    example_idx = tp_idx[0]
    print(f'Example: True Positive applicant (index {example_idx})')
    print(f'  Actual label     : {y_test_arr[example_idx]} (Default)')
    print(f'  Predicted prob   : {y_prob[example_idx]:.4f}')
    print(f'  Risk category    : HIGH')
else:
    example_idx = 0
    print(f'Using index 0 (no high-risk TP found at threshold 0.6)')
    print(f'  Predicted prob: {y_prob[0]:.4f}')

In [ ]:
# Build explainer and compute local explanation
explainer = build_explainer(model, X_train_tf, feature_names, model_name=model_name)
x_single  = X_test_tf[example_idx:example_idx+1]
local_exp = explain_single(explainer, x_single, feature_names, top_n=10)

print(f'\nLocal SHAP Explanation:')
print(f'  Base value  : {local_exp["base_value"]:.4f}')
print(f'  SHAP total  : {local_exp["prediction"]:.4f}  (log-odds space)')
print(f'  Model prob  : {y_prob[example_idx]:.4f}')

print('\nTop Risk-INCREASING factors:')
for c in local_exp['top_increasing'][:5]:
    print(f'  {c["feature"]:45s} val={c["feature_value"]:8.3f}  SHAP={c["shap_value"]:+.4f}')

print('\nTop Risk-REDUCING factors:')
for c in local_exp['top_decreasing'][:5]:
    print(f'  {c["feature"]:45s} val={c["feature_value"]:8.3f}  SHAP={c["shap_value"]:+.4f}')

## 5. Probability Calibration Results

In [ ]:
cal_path = Path('../models/calibration_results.json')
if cal_path.exists():
    with open(cal_path) as f:
        cal = json.load(f)
    print('=== Calibration Results ===')
    print(f'  Raw Brier Score        : {cal["brier_raw"]}')
    print(f'  Calibrated Brier Score : {cal["brier_calibrated"]}')
    print(f'  Improvement            : {cal["improvement"]} (positive = calibration helps)')
    print(f'  Calibration Applied    : {cal["calibration_applied"]}')
    print(f'  Conclusion: {cal["conclusion"]}')
else:
    print('calibration_results.json not found.')

In [ ]:
cal_fig = FIGURES_DIR / 'calibration_comparison.png'
if cal_fig.exists():
    display(Image(str(cal_fig)))

## 6. Fairness Analysis

In [ ]:
for attr in ['age', 'dependents']:
    p = Path(f'../reports/results/fairness_{attr}.csv')
    if p.exists():
        df_f = pd.read_csv(p)
        print(f'\n=== Fairness by {attr.upper()} ===')
        display(df_f.style.format({c: '{:.4f}' for c in df_f.select_dtypes('float').columns}))

print('\n=== Fairness Interpretation Notes ===')
print('1. Younger borrowers (<30) show higher default rates (11.9%) and higher FPR (30.7%)')
print('2. Older borrowers (>60) show lower recall (56.5%) — model misses more of their defaults')
print('3. ROC-AUC is similar across age groups (~0.84–0.87) — overall discrimination is consistent')
print('4. "Has Dependents" group shows higher selection rate (39.3%) vs no dependents (29.2%)')
print()
print('IMPORTANT LIMITATION:')
print('  These are model-level statistical disparities, NOT proof of bias or discrimination.')
print('  Age and dependents are proxy attributes, not directly observed protected characteristics.')
print('  Disparities may reflect historical lending patterns in the data, not model error.')

In [ ]:
for attr in ['age_group', 'dependents_group']:
    p = FIGURES_DIR / f'fairness_{attr}.png'
    if p.exists():
        display(Image(str(p)))

## 7. Summary of Research Question Answers

| RQ | Answer |
|---|---|
| RQ3: Calibration | Platt scaling improved Brier score by 0.0636 (0.1134→0.0498). **Calibrated model used.** |
| RQ4: Top features | total_past_due, RevolvingUtilization, age, DebtRatio, credit_line_density |
| RQ5: Local SHAP | Yes — per-applicant signed contributions clearly identify risk/safety factors |
| RQ6: Fairness | Measurable disparities in recall across age groups. Older borrowers show lower recall. |
| RQ7: Perf vs Interp | XGBoost (AUC=0.8599) outperforms Logistic Regression (AUC=0.8528) with similar recall. Trade-off is modest. |